# 🔬 Lesson 13 — Binding Site Investigation

### Analyzing Protein-Ligand Interactions with Python

---

## 📋 Project Overview

**Objective:** Build a complete pipeline to analyze the binding site 
of **Trypsin** in complex with the ligand **13U** — from downloading 
the structure from the PDB, through processing and preparation, 
to interaction analysis and visualization.

---

## 🎯 Learning Objectives

By the end of this lesson, we will be able to:

1. **Download** protein structures from the PDB programmatically
2. **Process** structures using MDAnalysis
3. **Handle** altLoc (alternate location) scenarios in high-resolution structures
4. **Visualize** the binding site interactively with NGLView
5. **Prepare** the protein (PDB2PQR) and ligand (RDKit + OpenBabel)
6. **Analyze** interactions using ProLIF
7. **Generate** a 2D interaction map (LigNetwork)

---

## 🛠️ Technical Stack

| Category | Library | Purpose |
|----------|---------|---------|
| **Molecular Processing** | MDAnalysis | Structure handling and atom selections |
| **3D Visualization** | NGLView | Interactive structure visualization |
| **Chemistry** | RDKit | Bond orders and SDF handling |
| **Hydrogen Addition** | PDB2PQR + OpenBabel | Adding hydrogens to protein & ligand |
| **Interaction Analysis** | ProLIF | Protein-ligand interaction fingerprinting |

---

## 🧬 Scientific Context

**Trypsin** is a serine protease that cleaves proteins at lysine 
and arginine residues.

**13U** is a synthetic inhibitor bound to the active site.

**PDB ID:** `2ZQ2` — Trypsin structure with ligand 13U bound.

**Medical Relevance:** Understanding the binding mechanism helps 
in designing drugs for pancreatitis and digestive disorders.

---

## 📅 Environment

- **Python:** 3.x
- **JupyterLab:** 4.6.4
- **Course:** MolSSI — Python Scripting for Biochemistry
- **Author:** Omar Hamouda


In [ ]:
# ═══════════════════════════════════════════════════════════
#  Imports & Environment Setup
# ═══════════════════════════════════════════════════════════

# ─── Standard Library ───
import os
import math
import requests

# ─── Scientific Stack ───
import numpy as np
import pandas as pd

# ─── Molecular Analysis ───
import MDAnalysis as mda

# ─── 3D Visualization (NGLView for JupyterLab 4) ───
import nglview as nv

# ─── Version Report ───
print("=" * 55)
print("  Binding Site Investigation Pipeline")
print("  Environment: JupyterLab 4.6.4")
print("=" * 55)
print(f"  MDAnalysis:  {mda.__version__}")
print(f"  NGLView:     {nv.__version__}")
print(f"  Pandas:      {pd.__version__}")
print(f"  NumPy:       {np.__version__}")
print("=" * 55)
print("  Environment ready")
print("=" * 55)

---

# 📥 Step 1 — Downloading the Protein Structure

## 🎯 Objective

Download the crystal structure of **Trypsin** in complex with 
the inhibitor **13U** directly from the RCSB Protein Data Bank.

## 🧬 Target Structure

| Property | Value |
|----------|-------|
| **PDB ID** | `2ZQ2` |
| **Protein** | Trypsin (Serine Protease) |
| **Ligand** | 13U (Synthetic Inhibitor) |
| **Method** | X-ray Crystallography |
| **Resolution** | High-resolution |

## 📡 Strategy

We will use the `requests` library to fetch the structure file 
programmatically — this is the **standard professional approach** 
in computational biochemistry, replacing manual downloads.




In [1]:
# Auto-Fix Working Directory 
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory set to:", os.getcwd())

Working directory set to: /home/omar/biochemist-python_ORGANIZED


In [ ]:
# ═══════════════════════════════════════════════════════════
#  Step 1: Download PDB Structure (2ZQ2)
# ═══════════════════════════════════════════════════════════

# ─── Setup ───
pdb_id = "2zq2"
pdb_dir = "pdb"

# Create directory for PDB files (safe if exists)
os.makedirs(pdb_dir, exist_ok=True)
print(f"Working directory: {os.path.abspath('.')}")
print(f"PDB folder:        {os.path.abspath(pdb_dir)}")
print()

# ─── Build URL ───
url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
pdb_path = os.path.join(pdb_dir, f"{pdb_id}.pdb")

# ─── Check if already downloaded (idempotent) ───
if os.path.exists(pdb_path):
    print(f"File already exists: {pdb_path}")
    print(f"Size: {os.path.getsize(pdb_path):,} bytes")
    print("Skipping download.")
else:
    print(f"Downloading from: {url}")
    print()
    
    # Send HTTP request
    response = requests.get(url)
    print(f"Status code: {response.status_code}")
    
    if response.status_code == 200:
        print("Download successful")
        
        # Save to file
        with open(pdb_path, "w+") as f:
            f.write(response.text)
        
        # Verify file
        file_size = os.path.getsize(pdb_path)
        print()
        print(f"Saved to:  {pdb_path}")
        print(f"File size: {file_size:,} bytes")
        print(f"Lines:     {len(response.text.splitlines()):,}")
    else:
        print(f"Download failed — status {response.status_code}")

---

# 🔍 Step 2 — Loading and Exploring the Structure

## 🎯 Objective

Load the downloaded PDB file into **MDAnalysis** and perform an 
initial exploration of its contents — the protein, the ligand, 
and the overall system composition.

## 🧬 What is MDAnalysis?

**MDAnalysis** is a Python library designed for analyzing molecular 
structures and dynamics trajectories. Its central object is the 
**Universe** — an abstraction of a molecular system containing 
topology (atoms, bonds, residues) and coordinates.

## 🔑 Key Concepts

| Concept | Description |
|---------|-------------|
| **Universe** | Complete molecular system (structure + coordinates) |
| **AtomGroup** | A selected subset of atoms |
| **Selection Language** | Human-readable syntax for isolating atoms |
| **altLoc** | Alternate locations for atoms in high-resolution structures |

## 📋 Exploration Plan

1. Load the PDB file into a Universe
2. Inspect basic system properties (atoms, residues, chains)
3. Identify the protein component
4. Identify the ligand component
5. Identify the water molecules

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Step 2: Load Structure with MDAnalysis
# ═══════════════════════════════════════════════════════════

# ─── Load Universe ───
u = mda.Universe(pdb_path)

print("=" * 60)
print("  System Overview")
print("=" * 60)

# ─── Overall Composition ───
print(f"  Total atoms:     {u.atoms.n_atoms:,}")
print(f"  Total residues:  {u.residues.n_residues:,}")
print(f"  Total segments:  {u.segments.n_segments}")
print()

# ─── Segments Info ───
print("Segments:")
for seg in u.segments:
    n_atoms = len(seg.atoms)
    n_residues = len(seg.residues)
    print(f"  - {seg.segid}: {n_atoms:,} atoms, {n_residues} residues")
print()

# ─── Protein ───
protein = u.select_atoms("protein")
print(f"Protein:         {protein.n_atoms:,} atoms, {protein.n_residues} residues")
print()

# ─── Ligand ───
ligand = u.select_atoms("resname 13U")
print(f"Ligand 13U:      {ligand.n_atoms} atoms, {ligand.n_residues} residue(s)")
print()

# ─── Water ───
water = u.select_atoms("resname HOH or resname WAT")
print(f"Water molecules: {water.n_residues:,}")
print()

print("=" * 60)
print("  Structure loaded successfully")
print("=" * 60)

---

# 👁️ Step 3 — Viewing the Structure in 3D

## 🎯 Objective

Before any analysis, let's **look at the structure** using 
**NGLView** — an interactive 3D molecular viewer for JupyterLab.

## 🔑 Why Visualize First?

- See the **overall system** (protein + ligand + water)
- Understand **why selections are necessary** — the raw view is messy
- Observe the **binding site** in context

## 🛠️ Helper Functions

We will define two reusable functions:

| Function | Purpose |
|----------|---------|
| `rotate_view(view, x, y, z)` | Orient the view for best angle |
| `view_binding_site(protein, ligand)` | Render surface + ligand |

## 📋 Plan

1. **Preview** the whole system (messy view)
2. **Select** protein + ligand
3. **Visualize** the binding site with hydrophobic surface

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Step 3: NGLView Helper Functions + Whole-System Preview
# ═══════════════════════════════════════════════════════════

# ─── Helper 1: Rotate the view ───
def rotate_view(view, x=0, y=0, z=0, degrees=True):
    """Rotate the NGLView widget along the given axes."""
    radians = 1
    if degrees:
        radians = math.pi / 180
    view.control.spin([1, 0, 0], x * radians)
    view.control.spin([0, 1, 0], y * radians)
    view.control.spin([0, 0, 1], z * radians)

# ─── Helper 2: Render binding site ───
def view_binding_site(protein, ligand):
    """Render binding site: hydrophobic surface + ligand."""
    view = nv.show_mdanalysis(protein)
    view.clear_representations()
    view.add_representation("surface", colorScheme="hydrophobicity")
    lig_view = view.add_component(ligand)
    lig_view.center()
    rotate_view(view, y=180, x=20)
    return view

# ─── Preview the WHOLE system (messy view) ───
view = nv.show_mdanalysis(u)
view

---

# 👁️ Step 4 — Visualizing the Full Ligand

## 🎯 Why Preview the Full Ligand?

Now let's select **the ligand WITHOUT altLoc filtering** and display it. 
It will appear **TWICE** — two overlapping conformations in the 
same binding pocket.

## 🔍 What to Look For

- **Two copies** of ligand 13U visible in 3D
- Overlapping chemical groups in different positions
- The **electron density** was ambiguous → both modeled

## 💡 Lesson

This **visual demonstration** motivates the next step: 
the **altLoc Analysis** — where we will decide which conformer 
to use for downstream processing.

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Step 4: View the Full Ligand (before altLoc filtering)
# ═══════════════════════════════════════════════════════════

# ─── Select protein + full ligand (A + B together) ───
ligand_all = u.select_atoms("resname 13U")

print(f"Full ligand atoms: {ligand_all.n_atoms}")
print("  → Expected: ~30 atoms (single conformer)")
print(f"  → Observed: {ligand_all.n_atoms} atoms (A + B together)")
print()

# ─── Display — ligand should appear TWICE ───
view = nv.show_mdanalysis(protein)
view.clear_representations()
view.add_representation("surface", colorScheme="hydrophobicity")
lig_view = view.add_component(ligand_all)
lig_view.center()
view

---

# ⚠️ Step 5 — Detecting and Handling altLoc

## 🎯 What is altLoc?

**altLoc** (Alternate Location) is a PDB convention that allows 
crystallographers to model **multiple possible positions** for an 
atom when the electron density is ambiguous.

In high-resolution crystal structures, some atoms — especially 
those in flexible regions or ligands — may be modeled in **two or 
more conformations**.

## 🔍 Why Does It Matter?

When you load a structure with altLoc records, MDAnalysis will 
**include ALL alternate conformations**, effectively **doubling** 
(or tripling) the atom count.

For our ligand:
- **Expected atoms:** ~30
- **Observed atoms:** 60
- **Conclusion:** The ligand exists in 2 alternate conformations

## 🔎 Detection Strategy

1. Inspect the `altLoc` attribute of each atom
2. Count atoms per altLoc identifier
3. Determine which conformation(s) to keep

## 📖 MDAnalysis Selection Syntax

```python
# Select all atoms with alternate location "A"
u.select_atoms("altLoc A")

# Select ligand in conformation A only
u.select_atoms("resname 13U and altLoc A")

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Step 5: Detect and Handle altLoc
# ═══════════════════════════════════════════════════════════

print("=" * 60)
print("  altLoc Analysis")
print("=" * 60)

# ─── Inspect altLoc identifiers in the ligand ───
ligand_all = u.select_atoms("resname 13U")

altloc_counts = {}
occupancy_by_altloc = {}

for atom in ligand_all:
    al = atom.altLoc if atom.altLoc else " "   # blank = no altLoc
    altloc_counts[al] = altloc_counts.get(al, 0) + 1
    occupancy_by_altloc.setdefault(al, []).append(atom.occupancy)

print()
print("Ligand 13U — Atom distribution by altLoc:")
print("-" * 60)
for al, count in sorted(altloc_counts.items()):
    avg_occ = np.mean(occupancy_by_altloc[al])
    label = al if al != " " else "(blank)"
    print(f"  altLoc = {label!r:>7}  →  {count:>3} atoms   (avg occupancy: {avg_occ:.2f})")

print()
print("=" * 60)

# ─── Split the ligand into conformers ───
ligand_A = u.select_atoms("resname 13U and altLoc A")
ligand_B = u.select_atoms("resname 13U and altLoc B")

print()
print("Conformer Selection:")
print("-" * 60)
print(f"  Ligand A (altLoc A): {ligand_A.n_atoms} atoms")
print(f"  Ligand B (altLoc B): {ligand_B.n_atoms} atoms")
print()
print("=" * 60)

---

# 🔍 Step 6 — Visualizing Both Conformers

## 🎯 Objective

Display the binding site with each conformer **separately** — 
to compare their positions and understand the differences.

## 🔍 What to Compare

| Conformer | Occupancy | Interpretation |
|-----------|-----------|----------------|
| **A** | 0.61 | Dominant conformation (higher probability) |
| **B** | 0.39 | Minor alternative conformation |

## 💡 Why Both?

Even though we'll use **Conformer A** for the interaction analysis 
(higher occupancy = more reliable), visualizing **both** helps us:

- Understand the **flexibility** of the binding pose
- See **alternate arrangements** of the ligand's chemical groups
- Appreciate the **crystallographer's decision** to model both

## 📋 Plan

1. Display binding site with **ligand_A**
2. Display binding site with **ligand_B**
3. Compare their positions in the pocket

---

# 🔍 Step 5 — Isolate Protein and Ligand

## 🎯 MDAnalysis Selection Syntax

MDAnalysis provides a **human-readable selection language** to 
isolate components from the Universe:

| Selection | Meaning |
|-----------|---------|
| `"protein"` | All protein atoms |
| `"resname 13U"` | Ligand atoms by residue name |
| `"resname 13U and altLoc A"` | Ligand in Conformer A only |

## 🎯 Plan

1. Select protein and ligand (Conformer A)
2. Save selections to PDB files (for downstream tools)
3. Visualize with NGLView: **surface + hydrophobicity**

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Step 6: Visualize Conformer A and Conformer B
# ═══════════════════════════════════════════════════════════

# ─── Display Conformer A (dominant) ───
print("Conformer A — Occupancy 0.61 (dominant)")
view_binding_site(protein, ligand_A)

In [ ]:
# ─── Display Conformer B (minor) ───
print("Conformer B — Occupancy 0.39 (minor)")
view_binding_site(protein, ligand_B)

---

# 💾 Step 7 — Saving Selections for Downstream Tools

## 🎯 Objective

Save the **protein** and **ligand A** (the dominant conformer) as 
separate PDB files. These will be used by external tools:

| File | Used By |
|------|---------|
| `protein_2zq2.pdb` | **PDB2PQR** (add hydrogens to protein) |
| `ligand_A.pdb` | **RDKit** (correct bond orders) |

## 🔑 Why Save to Files?

Different tools require **different inputs**:

- **PDB2PQR** → command-line tool → needs a file path
- **RDKit** → Python library → can read from file
- **OpenBabel** → needs a file (works on SDF/PDB)

## 💡 Best Practice

Separating the **raw structure** from the **processed outputs** 
keeps our pipeline modular and reproducible.

## 📋 Plan

1. Write `protein_2zq2.pdb`
2. Write `ligand_A.pdb`
3. Verify the files exist with correct sizes

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Step 7: Save Protein and Ligand A to PDB Files
# ═══════════════════════════════════════════════════════════

# ─── Define output paths ───
protein_path = os.path.join("pdb", "protein_2zq2.pdb")
ligand_path  = os.path.join("pdb", "ligand_A.pdb")

# ─── Write selections ───
protein.write(protein_path)
ligand_A.write(ligand_path)

# ─── Report ───
print(f"Saved protein: {protein_path}  ({os.path.getsize(protein_path):,} bytes)")
print(f"Saved ligand:  {ligand_path}   ({os.path.getsize(ligand_path):,} bytes)")

---

# 🧪 Step 8 — Protein Preparation with PDB2PQR

## 🎯 Why Add Hydrogens?

X-ray crystallography **cannot resolve hydrogen atoms** (too small 
to appear in electron density maps). But hydrogens are essential for:

- **Hydrogen bond detection** (ProLIF requires them)
- **Electrostatics calculations**
- **Molecular dynamics simulations**

## 🛠️ What is PDB2PQR?

**PDB2PQR** is a specialized tool for preparing biomolecules:

| Feature | Benefit |
|---------|---------|
| **Adds hydrogens** | At physiologically correct positions |
| **Repairs missing atoms** | Completes incomplete residues |
| **Optimizes H-bond network** | Ensures chemically sensible geometry |
| **Handles altLoc** | Chooses best occupancy for protein |





## 📖 Why pH = 7.4?

Physiological pH determines the **charge state** of ionizable residues:

| Residue | State at pH 7.4 |
|---------|-----------------|
| **Asp, Glu** | Negative (COO⁻) |
| **Lys, Arg** | Positive (NH₃⁺) |
| **His** | Neutral (uncharged) |
| **Cys** | Neutral (SH) |

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Step 8: Add Hydrogens to Protein with PDB2PQR
# ═══════════════════════════════════════════════════════════

# ─── Run PDB2PQR via command line ───
! pdb2pqr --pdb-output=pdb/protein_h.pdb --pH=7.4 pdb/protein_2zq2.pdb protein.pqr

---

# 🧬 Step 9 — Ligand Bond Order Correction (RDKit)

## 🎯 The Problem

PDB files **do not encode bond orders** accurately:
- Double bonds are recorded as single
- Aromatic rings lose their aromaticity
- Ring systems become chemically incorrect

This breaks **ProLIF** — which identifies interactions based 
on chemistry.

## 🛠️ The Solution

Use **RDKit** with a **reference structure** (`13U.sdf` from 
PubChem) to **transfer correct bond orders** onto our PDB ligand.




## 🎯 Key Steps

1. Load the reference ligand (`13U.sdf`)
2. Load the PDB ligand (`ligand_A.pdb`)
3. Remove hydrogens from reference (required by RDKit)
4. Transfer bond orders
5. Save as SDF (ready for hydrogen addition)

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Step 9: Fix Ligand Bond Orders with RDKit
# ═══════════════════════════════════════════════════════════
from rdkit import Chem
from rdkit.Chem.AllChem import AssignBondOrdersFromTemplate

# ─── Verify reference ligand exists ───
template_path = os.path.join("ligands", "13U.sdf")
if not os.path.exists(template_path):
    raise FileNotFoundError(f"Reference not found: {template_path}")

print(f"Reference:    {template_path}  ({os.path.getsize(template_path):,} bytes)")

# ─── Load the ideal ligand (reference) ───
template = Chem.MolFromMolFile(template_path, removeHs=False, sanitize=True)
template = Chem.RemoveAllHs(template)   # force remove ALL hydrogens
print(f"Template mol: {template.GetNumAtoms()} atoms, {template.GetNumBonds()} bonds")

# ─── Load the PDB ligand (from MDAnalysis) ───
pdb_ligand = Chem.MolFromPDBFile(ligand_path, removeHs=False, sanitize=False)
print(f"PDB ligand:   {pdb_ligand.GetNumAtoms()} atoms, {pdb_ligand.GetNumBonds()} bonds")

# ─── Transfer bond orders ───
ligand = AssignBondOrdersFromTemplate(template, pdb_ligand)
print(f"Final ligand: {ligand.GetNumAtoms()} atoms, {ligand.GetNumBonds()} bonds")

# ─── Save as SDF ───
sdf_path = os.path.join("pdb", "ligand.sdf")
Chem.MolToMolFile(ligand, sdf_path)

print()
print(f"Saved:        {sdf_path}  ({os.path.getsize(sdf_path):,} bytes)")

---

# 🧬 Step 7 — Ligand Bond Order Correction (RDKit)

## 🎯 The Problem
PDB files **don't encode bond orders** (double, aromatic) accurately — 
they're treated as single bonds. This breaks **ProLIF** analysis.

## 🛠️ The Solution
Use **RDKit** with a reference structure (`13U.sdf`) to **transfer 
correct bond orders** onto our PDB ligand.

## 🔄 Workflow
ligands/13U.sdf ← reference (correct chemistry)

+

pdb/ligand_A.pdb ← from MDAnalysis

│
▼

AssignBondOrdersFromTemplate()
│
▼

pdb/ligand.sdf ← corrected



In [ ]:
# ═══════════════════════════════════════════════════════════
#  Step 7: Fix Ligand Bond Orders with RDKit
# ═══════════════════════════════════════════════════════════
from rdkit import Chem
from rdkit.Chem.AllChem import AssignBondOrdersFromTemplate

# ─── Verify reference ligand exists ───
template_path = os.path.join("ligands", "13U.sdf")
if not os.path.exists(template_path):
    raise FileNotFoundError(f"Reference not found: {template_path}")

print(f"Reference:    {template_path}  ({os.path.getsize(template_path):,} bytes)")

# ─── Load the ideal ligand (reference) ───
template = Chem.MolFromMolFile(template_path, removeHs=False, sanitize=True)
template = Chem.RemoveAllHs(template)   # ← force remove ALL hydrogens
print(f"Template mol: {template.GetNumAtoms()} atoms, {template.GetNumBonds()} bonds")

# ─── Load the PDB ligand (from MDAnalysis) ───
pdb_ligand = Chem.MolFromPDBFile(ligand_path, removeHs=False, sanitize=False)
print(f"PDB ligand:   {pdb_ligand.GetNumAtoms()} atoms, {pdb_ligand.GetNumBonds()} bonds")

# ─── Transfer bond orders ───
ligand = AssignBondOrdersFromTemplate(template, pdb_ligand)
print(f"Final ligand: {ligand.GetNumAtoms()} atoms, {ligand.GetNumBonds()} bonds")

# ─── Save as SDF ───
sdf_path = os.path.join("pdb", "ligand.sdf")
Chem.MolToMolFile(ligand, sdf_path)

print()
print(f"Saved:        {sdf_path}  ({os.path.getsize(sdf_path):,} bytes)")

---

# 💧 Step 10 — Adding Hydrogens to the Ligand (OpenBabel)

## 🎯 Why Add Hydrogens to the Ligand?

**ProLIF requires explicit hydrogens** to detect H-bonds. The ligand 
from RDKit has **correct bond orders** but **no hydrogens**.

## 🛠️ Tool: OpenBabel

We will use **OpenBabel** (via `pybel`) — a cheminformatics toolkit 
that:
- Reads the SDF file from RDKit
- Adds hydrogens at chemically correct positions
- Writes a PDB file ready for ProLIF


## 📋 Plan

1. Read the SDF file
2. Add hydrogens using `addh()`
3. Write to PDB format


In [ ]:
# ═══════════════════════════════════════════════════════════
#  Step 10: Add Hydrogens to Ligand (OpenBabel)
# ═══════════════════════════════════════════════════════════
from openbabel import pybel

# ─── Input / Output paths ───
sdf_in  = os.path.join("pdb", "ligand.sdf")
pdb_out = os.path.join("pdb", "ligand_h.pdb")

# ─── Read SDF ───
mol = next(pybel.readfile("sdf", sdf_in))
print(f"Loaded:       {sdf_in}")
print(f"Atoms before: {len(mol.atoms)}")

# ─── Add hydrogens ───
mol.addh()

# ─── Write PDB ───
mol.write("pdb", pdb_out, overwrite=True)
print(f"Atoms after:  {len(mol.atoms)}")
print()
print(f"Saved:        {pdb_out}  ({os.path.getsize(pdb_out):,} bytes)")

---

# 🔬 Step 11 — Protein-Ligand Interaction Analysis (ProLIF)

## 🎯 What is ProLIF?

**ProLIF** = **Pro**tein-**L**igand **I**nteraction **F**ingerprints.

It detects interactions between a ligand and its binding site:

| Interaction Type | Description |
|------------------|-------------|
| **HBDonor** | Ligand donates a hydrogen bond |
| **HBAcceptor** | Ligand accepts a hydrogen bond |
| **VdWContact** | Van der Waals contact |
| **Hydrophobic** | Hydrophobic interaction |
| **PiStacking** | Aromatic ring stacking |
| **Cationic** | Interaction with positive charge |
| **Anionic** | Interaction with negative charge |
| **Metal** | Coordination with metal ion |

## 🎯 Requirements

- Protein **with hydrogens** (`protein_h.pdb`) ✅
- Ligand **with hydrogens** (`ligand_h.pdb`) ✅
- Correct bond orders ✅


## 📋 Plan

1. Load both prepared files
2. Convert to ProLIF molecules
3. Run interaction fingerprinting
4. Display results as a summary table


In [ ]:
# ═══════════════════════════════════════════════════════════
#  Step 11: Run ProLIF Interaction Analysis
# ═══════════════════════════════════════════════════════════
import prolif as plf

# ─── Reload prepared files ───
protein_h = mda.Universe("pdb/protein_h.pdb")
ligand_h  = mda.Universe("pdb/ligand_h.pdb")

print(f"Protein H: {protein_h.atoms.n_atoms:,} atoms")
print(f"Ligand H:  {ligand_h.atoms.n_atoms:,} atoms")
print()

# ─── Convert to ProLIF molecules ───
protein_mol = plf.Molecule.from_mda(protein_h)
ligand_mol  = plf.Molecule.from_mda(ligand_h)

# ─── Create fingerprint and run ───
fp = plf.Fingerprint()
lig_list = [ligand_mol]
interactions = fp.run_from_iterable(lig_list, protein_mol)

print("=" * 60)
print("  ProLIF analysis complete")
print("=" * 60)
print()
print(f"Interaction object type: {type(interactions).__name__}")

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Step 11b: Display ProLIF Results
# ═══════════════════════════════════════════════════════════

# ─── Convert to DataFrame ───
df = interactions.to_dataframe()

print("=" * 60)
print("  Interaction Summary — Trypsin + 13U")
print("=" * 60)

# ─── Interaction types (from MultiIndex columns) ───
interaction_types = df.columns.get_level_values("interaction")
print()
print("Interaction types detected:")
print(interaction_types.value_counts().to_string())

# ─── Protein residues involved ───
residues = df.columns.get_level_values("protein")
unique_residues = sorted(set(residues))

print()
print("Protein residues involved:")
for res in unique_residues:
    print(f"  - {res}")

print()
print(f"Total unique interactions: {len(df.columns)}")
print(f"Unique protein residues:   {len(unique_residues)}")

---

# 🗺️ Step 12 — Interaction Network (LigNetwork)

## 🎯 What is LigNetwork?

An **interactive 2D map** that shows:

| Element | Description |
|---------|-------------|
| **Ligand** | Central node with atom-level details |
| **Residues** | Protein residues as colored boxes |
| **Edges** | Interactions between them |

## 🎯 Why It Matters

- **One-glance overview** of all interactions
- **Publication-quality** figure
- **Faster** than reading a table
- **Color-coded** by interaction type

## 🎨 Color Legend

| Color | Interaction Type |
|-------|------------------|
| 🟠 Orange | Van der Waals Contact |
| 🔵 Blue | Hydrogen Bond (Donor/Acceptor) |
| 🟣 Purple | Aromatic (π-Stacking) |
| 🔴 Red | Acidic residue |
| 🟢 Green | Aliphatic residue |

## 📋 Plan

1. Generate the LigNetwork from the fingerprint object
2. Explore interactions interactively
3. Identify key binding residues

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Step 12: Generate 2D Interaction Map (LigNetwork)
# ═══════════════════════════════════════════════════════════

# ─── Display the LigNetwork widget ───
view = fp.plot_lignetwork(lig_list[0])
view

---

# ✅ Summary — Lesson 13 Complete

## 🎯 What We Accomplished

1. **Downloaded** the Trypsin-13U complex from PDB (2ZQ2)
2. **Processed** the structure with MDAnalysis
3. **Visualized** the whole system (messy view → motivation for selections)
4. **Handled** altLoc conformers (selected A, highest occupancy)
5. **Visualized** both conformers (A and B) for comparison
6. **Prepared** protein with PDB2PQR (added hydrogens)
7. **Corrected** ligand bond orders with RDKit
8. **Added** hydrogens to ligand with OpenBabel
9. **Analyzed** interactions with ProLIF
10. **Generated** 2D interaction map (LigNetwork)

---

## 🔬 Key Findings

| Metric | Value |
|--------|-------|
| **Total interactions** | 17 |
| **Protein residues involved** | 12 |
| **Van der Waals contacts** | 12 |
| **H-bond donors** | 3 |
| **H-bond acceptors** | 2 |

### 🔑 Key Residues

| Residue | Interaction | Role |
|---------|-------------|------|
| **ASP189** | HBDonor + VdW | S1 pocket anchor |
| **GLY216** | HBAcceptor + VdW | Oxyanion hole region |
| **GLY219** | HBDonor + VdW | Active site vicinity |
| **SER190** | VdW | S1 pocket |
| **TRP215** | VdW | Aromatic box |

---

## 🧬 Scientific Insight

The ligand **13U** binds to the **S1 pocket** of Trypsin — a 
well-known binding site for arginine/lysine-containing substrates. 
The interaction profile reveals:

- **ASP189** (negatively charged) forms a key H-bond — 
  characteristic of Trypsin's specificity for positively charged residues
- Multiple **VdW contacts** stabilize the binding pose
- The **hydrophobic surface** of the pocket complements the ligand's 
  aromatic ring

---

## 🛠️ Technical Stack

| Tool | Purpose |
|------|---------|
| **MDAnalysis** | Structure processing & selections |
| **NGLView** | 3D visualization (hydrophobic surface) |
| **PDB2PQR** | Protein hydrogen addition |
| **RDKit** | Bond order correction |
| **OpenBabel** | Ligand hydrogen addition |
| **ProLIF** | Interaction fingerprinting |

---

## 📁 Output Files

| File | Description |
|------|-------------|
| `pdb/2zq2.pdb` | Original PDB |
| `pdb/protein_2zq2.pdb` | Protein from MDAnalysis |
| `pdb/ligand_A.pdb` | Ligand Conformer A |
| `pdb/protein_h.pdb` | Protein + hydrogens |
| `pdb/ligand.sdf` | Corrected ligand (bond orders) |
| `pdb/ligand_h.pdb` | Ligand + hydrogens |

---

**End of Lesson 13 — Binding Site Investigation** 🎓